In [2]:
%pip install beautifulsoup4 requests
# 연속으로 설치하면 좋은 점
# 서로 의존성이 있는 패키지들을 한꺼번에 설치할 수 있다.
# 그래서 라이브러리를 설치할 때는 한꺼번에 설치하는 것이 좋다.
# 우와 신기해

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# 라이브러리 불러오기
from bs4 import BeautifulSoup
import requests
import re
# re는 정규 표현식(regular expression) 라이브러리
import pandas as pd

In [ ]:
# 위키피디아 미국 ETF 웹 페이지에서 필요한 정보를 스크래핑하여 딕셔너리 형태로 변수 etfs에 저장
# 크롤링 정책 중에 User-Agent
# 위키피디아 미국 ETF 웹 페이지에서 F12를 켜서 div 밑에 ul 밑에 li를 가져온다
url = "https://en.wikipedia.org/wiki/List_of_American_exchange-traded_funds"
header = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/92.0.4515.159 Safari/537.36'
    }

resp = requests.get(url, headers=header)
soup = BeautifulSoup(resp.text, 'lxml')   
rows = soup.select('div > ul > li')

print("resp: ", resp)
print("=" * 100)
print("resp.text: ", resp.text)
print("=" * 100)
print("soup: ", soup)
print("=" * 100)
print("rows: ", rows[50:55])

resp:  <Response [403]>
resp.text:  Please set a user-agent and respect our robot policy https://w.wiki/4wJS. See also T400119.

soup:  <html><body><p>Please set a user-agent and respect our robot policy https://w.wiki/4wJS. See also T400119.
</p></body></html>
rows:  []


In [10]:
# 크롤링 가능 여부 확인
import urllib.robotparser
rp = urllib.robotparser.RobotFileParser()
rp.set_url("https://wikitech.wikimedia.org/robots.txt")
rp.read()
url = 'https://wikitech.wikimedia.org'

In [7]:
# 위키피디아 미국 ETF 웹 페이지에서 필요한 정보를 스크래핑하여 딕셔너리 형태로 변수 etfs에 저장
url = "https://en.wikipedia.org/wiki/List_of_American_exchange-traded_funds"
header = {
"User-Agent": "CoolBot/0.0 (https://example.org/coolbot/; coolbot@example.org) generic-library/0.0"
}

resp = requests.get(url, headers=header )
soup = BeautifulSoup(resp.text, 'lxml')   
rows = soup.select('#mw-content-text > div.mw-content-ltr.mw-parser-output > ul:nth-child(11) > li') 
for row in rows:
    print(row.text)

iShares Core S&P Total US Stock Mkt (NYSE Arca: ITOT)
iShares MSCI ACWI Index (Nasdaq: ACWI)
iShares Russell 3000 Index (NYSE Arca: IWV)
Schwab US Broad Market ETF (NYSE Arca: SCHB)
Schwab Fundamental U.S. Broad Market Index ETF (NYSE Arca: FNDB)
Vanguard Total World Stock (NYSE Arca: VT), tracks the FTSE All-World Index
Vanguard Total Stock Market (NYSE Arca: VTI), tracks the MSCI US Broad Market Index
Vanguard Total International Stock (NYSE Arca: VXUS), tracks the MSCI All Country World ex-USA Investable Market Index
Vanguard Russell 3000 (NYSE Arca: VTHR), tracks 98% of the US market


In [5]:
etfs = {}
for row in rows:
    
    try:
        etf_name = re.findall("(.+?)\s\(", row.text)
        etf_market = re.findall("\((.+?):", row.text)
        etf_ticker = re.findall(":\s(.+)\)", row.text)
        
        if (len(etf_ticker) > 0) & (len(etf_market) > 0) & (len(etf_name) > 0):
            etfs[etf_ticker[0]] = [etf_market[0], etf_name[0]]

    except AttributeError as err:
        pass    

# etfs 딕셔너리 출력
print(etfs)

{}


<>:5: SyntaxWarning: invalid escape sequence '\s'
<>:6: SyntaxWarning: invalid escape sequence '\('
<>:7: SyntaxWarning: invalid escape sequence '\s'
<>:5: SyntaxWarning: invalid escape sequence '\s'
<>:6: SyntaxWarning: invalid escape sequence '\('
<>:7: SyntaxWarning: invalid escape sequence '\s'
C:\Users\sally\AppData\Local\Temp\ipykernel_38380\1232222225.py:5: SyntaxWarning: invalid escape sequence '\s'
  etf_name = re.findall("(.+?)\s\(", row.text)
C:\Users\sally\AppData\Local\Temp\ipykernel_38380\1232222225.py:6: SyntaxWarning: invalid escape sequence '\('
  etf_market = re.findall("\((.+?):", row.text)
C:\Users\sally\AppData\Local\Temp\ipykernel_38380\1232222225.py:7: SyntaxWarning: invalid escape sequence '\s'
  etf_ticker = re.findall(":\s(.+)\)", row.text)


In [6]:
# etfs 딕셔너리를 데이터프레임으로 변환
df = pd.DataFrame(etfs)
df

""
